<a href="https://colab.research.google.com/github/goldurroman/A62-Ideation/blob/main/01_2_Train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# INSTALLATION DES DÉPENDANCES
# ============================================================

!pip install ultralytics opencv-python scikit-image matplotlib numpy tqdm

print("[OK] Installation terminée.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 50.6 MB/s eta 0:00:00
[OK] Installation terminée.


In [ ]:
# @title
# ============================================================
# VÉRIFICATION DU GPU
# ============================================================

import torch

print("GPU disponible :", torch.cuda.is_available())
if torch.cuda.is_available():
    !nvidia-smi


GPU disponible : False


In [ ]:
# ============================================================
# MONTAGE GOOGLE DRIVE
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

print("[OK] Drive monté.")


Mounted at /content/drive
[OK] Drive monté.


In [ ]:
# ============================================================
# RESET COMPLET DU RUNTIME (SUPPRESSION DES ANCIENS FICHIERS)
# ============================================================

!rm -rf /content/isic_yolo
!rm -rf /content/isic2018_raw
!rm -rf /content/runs

print("[OK] Reset complet effectué.")


[OK] Reset complet effectué.


In [ ]:
# ============================================================
# EXTRACTION DES ZIP ISIC 2018
# ============================================================

import os

raw_dir = "/content/isic2018_raw"
os.makedirs(raw_dir, exist_ok=True)

zip_images = "/content/drive/MyDrive/ISIC2018_Task1-2_Training_Input.zip"
zip_masks  = "/content/drive/MyDrive/ISIC2018_Task1_Training_GroundTruth.zip"

!unzip -q "{zip_images}" -d "{raw_dir}"
!unzip -q "{zip_masks}"  -d "{raw_dir}"

train_input_dir = f"{raw_dir}/ISIC2018_Task1-2_Training_Input"
gt_dir          = f"{raw_dir}/ISIC2018_Task1_Training_GroundTruth"

print("[OK] Extraction terminée.")


[OK] Extraction terminée.


In [ ]:
# ============================================================
# MINI DATASET : 100 TRAIN / 20 VAL / 20 TEST
# ============================================================

import glob, random

all_imgs = sorted(glob.glob(train_input_dir + "/*.jpg"))
print("Total images disponibles :", len(all_imgs))

random.seed(42)
random.shuffle(all_imgs)

N_TRAIN = 550
N_VAL   = 50
N_TEST  = 50

train_imgs = all_imgs[:N_TRAIN]
val_imgs   = all_imgs[N_TRAIN:N_TRAIN+N_VAL]
test_imgs  = all_imgs[N_TRAIN+N_VAL:N_TRAIN+N_VAL+N_TEST]

print("[OK] Split mini-dataset :")
print("Train :", len(train_imgs))
print("Val   :", len(val_imgs))
print("Test  :", len(test_imgs))


Total images disponibles : 2594
[OK] Split mini-dataset :
Train : 750
Val   : 50
Test  : 50


In [ ]:
# ============================================================
# STRUCTURE YOLO
# ============================================================

base_dir = "/content/isic_yolo"

images_train = f"{base_dir}/images/train"
images_val   = f"{base_dir}/images/val"
images_test  = f"{base_dir}/images/test"

labels_train = f"{base_dir}/labels/train"
labels_val   = f"{base_dir}/labels/val"
labels_test  = f"{base_dir}/labels/test"

!mkdir -p "{images_train}" "{images_val}" "{images_test}" "{labels_train}" "{labels_val}" "{labels_test}"

print("[OK] Structure YOLO créée.")


[OK] Structure YOLO créée.


In [ ]:
# ============================================================
# COPIE DES IMAGES
# ============================================================

import shutil
from tqdm import tqdm

def copy_list(img_list, dest):
    for p in tqdm(img_list):
        shutil.copy(p, dest)

copy_list(train_imgs, images_train)
copy_list(val_imgs, images_val)
copy_list(test_imgs, images_test)

print("[OK] Copie terminée.")


100%|██████████| 50/50 [00:01<00:00, 27.04it/s]

[OK] Copie terminée.


In [ ]:
# ============================================================
# GÉNÉRATION DES LABELS YOLO SEGMENTATION
# ============================================================

import cv2
import numpy as np
from skimage import measure
import os
from tqdm import tqdm

def img_to_mask(img_path):
    name = os.path.basename(img_path).replace(".jpg", "_segmentation.png")
    return f"{gt_dir}/{name}"

def mask_to_yolo(mask_path, w, h, epsilon_factor=0.03, min_area_ratio=0.001):
    """
    - Simplification polygones
    - Nettoyage du masque
    - Suppression des petits contours
    """

    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if mask is None:
        print(" Masque introuvable :", mask_path)
        return []

    # 1) Lissage pour réduire les détails
    mask = cv2.GaussianBlur(mask, (7, 7), 0)

    # 2) Binarisation
    _, bin_mask = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)

    # 3) Petite érosion pour simplifier encore plus
    kernel = np.ones((5, 5), np.uint8)
    bin_mask = cv2.erode(bin_mask, kernel, iterations=1)

    # 4) Extraction des contours
    contours, _ = cv2.findContours(bin_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)

    polys = []
    img_area = w * h

    for cnt in contours:
        area = cv2.contourArea(cnt)

        # 5) Supprimer les petits contours
        if area < img_area * min_area_ratio:
            continue

        # 6) Simplification extrême
        epsilon = epsilon_factor * cv2.arcLength(cnt, True)
        approx = cv2.approxPolyDP(cnt, epsilon, True)

        if len(approx) < 3:
            continue

        # 7) Conversion en coordonnées YOLO
        poly = []
        for p in approx:
            x = p[0][0] / w
            y = p[0][1] / h
            poly.extend([x, y])

        if len(poly) >= 6:
            polys.append("0 " + " ".join([f"{v:.6f}" for v in poly]))

    return polys



def create_labels(img_list, label_dir):
    for img_path in tqdm(img_list):
        img = cv2.imread(img_path)
        if img is None:
            continue
        h, w = img.shape[:2]
        mask_path = img_to_mask(img_path)
        polys = mask_to_yolo(mask_path, w, h)
        label_path = os.path.join(label_dir, os.path.basename(img_path).replace(".jpg", ".txt"))
        with open(label_path, "w") as f:
            f.write("\n".join(polys))

create_labels(train_imgs, labels_train)
create_labels(val_imgs, labels_val)
create_labels(test_imgs, labels_test)

print("[OK] Labels générés.")


100%|██████████| 50/50 [00:12<00:00,  4.07it/s]

[OK] Labels générés.


In [ ]:
# ============================================================
# CRÉATION DU FICHIER data.yaml
# ============================================================

yaml = """
path: /content/isic_yolo
train: images/train
val: images/val
test: images/test

names:
  0: lesion
"""

with open("isic2018-seg.yaml", "w") as f:
    f.write(yaml)

print("[OK] YAML créé.")


[OK] YAML créé.


In [ ]:
import os
import glob
from datetime import datetime

def make_run_name(model):
    # Nom du modèle (ex: yolov8n-seg)
    model_name = os.path.basename(model.ckpt_path).replace(".pt", "")

    # Comptage automatique des images
    base = "/content/isic_yolo/images"
    train_n = len(glob.glob(f"{base}/train/*.jpg"))
    val_n   = len(glob.glob(f"{base}/val/*.jpg"))
    test_n  = len(glob.glob(f"{base}/test/*.jpg"))

    # Timestamp utile pour éviter les collisions
    timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")

    # Nom final
    return f"{model_name}-{train_n}-{val_n}-{test_n}-{timestamp}"


In [ ]:
# ============================================================
# ENTRAÎNEMENT YOLOv8n‑SEG (STABLE)
# ============================================================

from ultralytics import YOLO

model = YOLO("yolov8m-seg.pt")

run_name = make_run_name(model)

results = model.train(
    data="isic2018-seg.yaml",
    epochs=150, #Nombre total de passes complètes sur le dataset.
    imgsz=128, #Taille des images d’entrée (128×128).
    batch=1, #Nombre d’images traitées simultanément.
    workers=0, #Nombre de threads pour charger les données.
    augment=False, #augmentations d’images.False = plus stable, moins de bruit
    cache=False, #Stocke les images en RAM pour accélérer l’entraînement.
    deterministic=True, #Force la reproductibilité :
    name=run_name,
    patience=20, #Si la validation ne s’améliore pas pendant 20 epochs → arrêt
    lr0=0.0005, #Learning rate initial. Plus petit = plus stable
    optimizer="AdamW", #Optimiseur utilisé pour mettre à jour les poids.
    cos_lr=True, #Active le scheduler Cosine Annealing.meilleure convergence.
    lrf=0.05, #Learning rate final. stabilise la fin de l’entraînement.
    weight_decay=0.001, #Régularisation L2 pour éviter l’overfitting.
    mask_ratio=2, #Contrôle la pondération des masques dans la loss.
    dropout=0.05 #Améliore la généralisation
)



print("\n[OK] Entraînement terminé.")


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.48 🚀 Python-3.12.13 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=isic2018-seg.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.05, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=128, in

In [ ]:
# ============================================================
# FONCTION : SAUVEGARDE AUTOMATIQUE D'UN RUN YOLO DANS DRIVE
# ============================================================

import shutil
import os
import glob

def save_yolo_run_auto(results, model):
    """
    Sauvegarde automatique d'un run YOLO dans Google Drive.
    - Détecte le nom du modèle (ex: yolov8n-seg)
    - Détecte la taille du dataset (train/val/test)
    - Génère un nom dynamique (ex: yolov8n-seg-runs-100-20-20)
    - Copie le dossier complet dans Drive
    """

    # 1) Nom du modèle (ex: "yolov8n-seg.pt" → "yolov8n-seg")
    model_name = os.path.basename(model.ckpt_path).replace(".pt", "")

    # 2) Dossier du run YOLO
    run_dir = results.save_dir

    # 3) Comptage automatique des images
    base = "/content/isic_yolo/images"
    train_n = len(glob.glob(f"{base}/train/*.jpg"))
    val_n   = len(glob.glob(f"{base}/val/*.jpg"))
    test_n  = len(glob.glob(f"{base}/test/*.jpg"))

    # 4) Nom final du dossier
    folder_name = f"{model_name}-runs-{train_n}-{val_n}-{test_n}"
    dst = f"/content/drive/MyDrive/{folder_name}"

    # 5) Suppression si dossier déjà existant
    if os.path.exists(dst):
        shutil.rmtree(dst)

    # 6) Copie complète du run
    shutil.copytree(run_dir, dst)

    print(f"\n[OK] Run sauvegardé dans Drive : {dst}")
    return dst


In [ ]:
save_yolo_run_auto(results, model)
